## Installation (execute only if running on a cloud platform!)¶

In [5]:
# -- Use the following line for google colab removing the hash at the beginning.
! pip install -q 'corner==2.2.2' 'bilby==2.2.2' 'astropy==6.0.1'

## Initialization

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import corner

In [2]:
!pip install healpy astropy pyvo pandas

In [3]:
# REQUIREMENTS (already used in your notebook):
# !pip install pyvo astropy healpy pandas

import json, numpy as np, pandas as pd, healpy as hp
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u

# ---------- Config ----------
TAP_URL = "https://datalab.noirlab.edu/tap"
SKYMAP_URL = "https://dcc.ligo.org/LIGO-P2000230/public/GW190814_skymap.fits.gz"
CREDIBLE = 0.90
SQL_FATAL_QUALITY = True          # keep True to avoid multi-million row downloads
MAG_MIN, MAG_MAX = 16.0, 24.5     # light magnitude guard for tractability
NSIDE_FORCE = None                # set to an int to override skymap NSIDE if needed (usually None)
NEST_FORCE  = None                # set to True/False to override, else read from skymap

Z_MIN, Z_MAX = 0.043, 0.061       # from GW distance ~ 200–280 Mpc

from pyvo import dal as vo

svc = vo.TAPService(TAP_URL)

# ---------- Helpers ----------
def hpd_threshold(prob, level):
    flat = prob.ravel()
    order = np.argsort(flat)[::-1]
    csum = np.cumsum(flat[order])
    return flat[order[np.searchsorted(csum, level * flat.sum())]]

def healpix_components(mask, nside, nest=True):
    npix = mask.size
    visited = np.zeros(npix, bool)
    comps = []
    idxs = np.where(mask)[0]
    idxset = set(idxs.tolist())
    for s in idxs:
        if visited[s]: continue
        q=[int(s)]; visited[s]=True; comp=[int(s)]
        while q:
            p=q.pop()
            neigh = hp.get_all_neighbours(nside, p, nest=nest)
            for nb in neigh[neigh>=0]:
                if (nb in idxset) and (not visited[nb]):
                    visited[nb]=True; q.append(int(nb)); comp.append(int(nb))
        comps.append(comp)
    return comps

def minimal_ra_span(ra_deg):
    ra = np.mod(ra_deg, 360.0)
    s = np.sort(ra); dbl = np.concatenate([s, s+360])
    N=len(s); best=(1e9,0,0)
    for i in range(N):
        j=i+N-1
        w=dbl[j]-dbl[i]
        if w<best[0]:
            a=(dbl[i])%360.0; b=(a+w)%360.0; best=(w,a,b)
    return best[1], best[2]

# ---------- 1) Load skymap & build 90% main island ----------
with fits.open(SKYMAP_URL, memmap=True) as h:
    tab = h[1].data; hdr = h[1].header
    prob      = np.asarray(tab["PROB"], float)
    distmu    = np.asarray(tab["DISTMU"], float)
    distsigma = np.asarray(tab["DISTSIGMA"], float)
    nside = int(hdr["NSIDE"]) if NSIDE_FORCE is None else int(NSIDE_FORCE)
    nest  = hdr.get("ORDERING","NESTED").upper().startswith("NEST") if NEST_FORCE is None else bool(NEST_FORCE)

thr  = hpd_threshold(prob, CREDIBLE)
mask = prob >= thr
comps = healpix_components(mask, nside, nest=nest)
main  = np.array(comps[np.argmax([prob[c].sum() for c in comps])], dtype=int)

theta, phi = hp.pix2ang(nside, main, nest=nest)
dec_deg = 90.0 - np.degrees(theta)
ra_deg  = np.degrees(phi) % 360.0
dec_min, dec_max = float(dec_deg.min()), float(dec_deg.max())
ra_min, ra_max   = minimal_ra_span(ra_deg)

funnel = {}

# ---------- 2) SQL rectangle query (DES main ⨝ y6_gold), fatal-only limits ----------
# We *do not* apply morphology or photo-z in SQL to preserve the funnel order.
where_sky = f"""
(g.ra BETWEEN {ra_min} AND {ra_max}) AND
(g.dec BETWEEN {dec_min} AND {dec_max})
"""

where_quality = "1=1"
if SQL_FATAL_QUALITY:
    where_quality = f"(g.flags_i < 4) AND (g.mag_auto_i BETWEEN {MAG_MIN} AND {MAG_MAX})"

adql = f"""
SELECT
  g.coadd_object_id AS coadd_object_id,
  g.ra, g.dec,
  g.mag_auto_i, g.flags_i,
  y.dnf_z,
  y.wavg_spread_model_z, y.wavg_spreaderr_model_z,
  y.spread_model_z,      y.spreaderr_model_z
FROM des_dr2.main AS g
JOIN des_dr2.y6_gold AS y
  ON g.coadd_object_id = y.coadd_object_id
WHERE
  {where_sky}
  AND {where_quality}
"""

rect_tbl = svc.search(adql).to_table()
funnel["step1_rectangle_sql"] = int(len(rect_tbl))

print(f"[1] Rectangle (SQL) — rows: {funnel['step1_rectangle_sql']}  "
      f"[flags_i<4={SQL_FATAL_QUALITY}, mag {MAG_MIN}–{MAG_MAX}]")

# ---------- 3) Exact 90% main-island mask ----------
theta = np.radians(90.0 - np.array(rect_tbl["dec"]))
phi   = np.radians(np.array(rect_tbl["ra"]) % 360.0)
pix   = hp.ang2pix(nside, theta, phi, nest=nest)
in_island = np.isin(pix, main)
island_tbl = rect_tbl[in_island]
funnel["step2_inside_island"] = int(len(island_tbl))
print(f"[2] Inside exact 90% main island — rows: {funnel['step2_inside_island']}")

# ---------- 4) Morphology (galaxy-like) ----------
wok = (island_tbl["wavg_spread_model_z"] > -98) & (island_tbl["wavg_spreaderr_model_z"] > -98)
m3  = np.where(
    wok,
    island_tbl["wavg_spread_model_z"] + 3*island_tbl["wavg_spreaderr_model_z"],
    island_tbl["spread_model_z"]      + 3*island_tbl["spreaderr_model_z"]
)
gal_mask = m3 > 0.005
gal_tbl  = island_tbl[gal_mask]
funnel["step3_morph_galaxies"] = int(len(gal_tbl))
print(f"[3] Morphology (extended) — rows: {funnel['step3_morph_galaxies']}")

# ---------- 5) Photo-z window (distance slice) ----------
z_ok = (gal_tbl["dnf_z"] >= Z_MIN) & (gal_tbl["dnf_z"] <= Z_MAX)
final_tbl = gal_tbl[z_ok]
funnel["step4_photoz_window"] = int(len(final_tbl))
print(f"[4] Photo-z in [{Z_MIN},{Z_MAX}] — rows: {funnel['step4_photoz_window']}")

# ---------- Save artifacts ----------
# CSV + FITS of final candidates
final_df = final_tbl.to_pandas()
final_df.to_csv("GW190814_candidates_90pct.csv", index=False)
final_tbl.write("GW190814_candidates_90pct.fits", overwrite=True)

# Counts JSON + metadata
meta = {
    "event": "GW190814",
    "credible_level": CREDIBLE,
    "main_island_prob_mass": float(prob[main].sum()),
    "nside": nside, "nest": nest,
    "ra_min_deg": ra_min, "ra_max_deg": ra_max,
    "dec_min_deg": dec_min, "dec_max_deg": dec_max,
    "sql_fatal_quality": SQL_FATAL_QUALITY,
    "mag_range_i": [MAG_MIN, MAG_MAX],
    "z_window": [Z_MIN, Z_MAX]
}
out = {"counts": funnel, "meta": meta}
with open("GW190814_funnel_counts.json", "w") as f:
    json.dump(out, f, indent=2)

print("\nSaved:")
print(" - GW190814_candidates_90pct.csv")
print(" - GW190814_candidates_90pct.fits")
print(" - GW190814_funnel_counts.json")


[1] Rectangle (SQL) — rows: 2119644  [flags_i<4=True, mag 16.0–24.5]
[2] Inside exact 90% main island — rows: 1393743


[3] Morphology (extended) — rows: 1321738
[4] Photo-z in [0.043,0.061] — rows: 589

Saved:
 - GW190814_candidates_90pct.csv
 - GW190814_candidates_90pct.fits
 - GW190814_funnel_counts.json


In [9]:
# Crossmatch your 589 DES candidates to GLADE (GLADE+) via VizieR (robust)
# Outputs:
#  - GW190814_DES_vs_GLADE_matches_2arcsec.csv
#  - GW190814_DES_candidates_not_in_GLADE_2arcsec.csv

# ---------- deps ----------
import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split("==")[0].split(">=")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for p in ["astroquery", "astropy", "pandas", "numpy"]:
    pip_install(p)

import pandas as pd, numpy as np
from astropy.table import Table, hstack
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

# ---------- inputs ----------
des_csv = "/content/GW190814_candidates_90pct.csv"     # your 589-list
# 90% main-island rectangle (deg) – from your funnel run
ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484

margin_deg   = 0.15          # safety border around the box
match_radius = 2.0 * u.arcsec

# ---------- load DES candidates ----------
des_df = pd.read_csv(des_csv)
if not {"ra","dec"}.issubset({c.lower() for c in des_df.columns}):
    raise ValueError("Expected columns 'ra' and 'dec' in the CSV.")
des_tab = Table.from_pandas(des_df)

# ---------- find a GLADE catalog on this VizieR mirror ----------
cats = Vizier.find_catalogs("GLADE")
if not cats:
    raise RuntimeError("No GLADE entries found on this VizieR mirror.")
# Prefer the collection key containing 'VII/281' (GLADE+), else take any GLADE entry
preferred = [k for k in cats.keys() if "VII/281" in k]
cat_id = preferred[0] if preferred else list(cats.keys())[0]
print("Using catalog id:", cat_id)

# ---------- discover RA/Dec column names ----------
Vizier.ROW_LIMIT = 50
sample = None
try:
    sample = Vizier.get_catalogs(cat_id)[0]   # sometimes returns metadata table
except Exception:
    pass
# try a tiny region to get real columns if needed
if sample is None or len(sample.colnames) == 0:
    try:
        tiny = Vizier.query_region(SkyCoord(0*u.deg, 0*u.deg), width=0.1*u.deg, height=0.1*u.deg, catalog=cat_id)
        if len(tiny) > 0:
            sample = tiny[0]
    except Exception:
        pass

if sample is None:
    raise RuntimeError("Could not inspect columns for the chosen GLADE catalog.")

ra_candidates  = ["RAJ2000","RA_ICRS","_RAJ2000","raj2000","ra"]
dec_candidates = ["DEJ2000","DE_ICRS","_DEJ2000","dej2000","dec"]
cols_lower = {c.lower(): c for c in sample.colnames}

def pick_name(cands):
    for c in cands:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

glade_ra  = pick_name(ra_candidates)
glade_dec = pick_name(dec_candidates)
if glade_ra is None or glade_dec is None:
    raise RuntimeError(f"Could not find RA/Dec column names. Columns seen: {sample.colnames}")
print("GLADE RA/Dec columns:", glade_ra, glade_dec)

# ---------- region query for our rectangle (+ margin) ----------
Vizier.ROW_LIMIT = -1
Vizier.columns = [glade_ra, glade_dec, "Name", "z", "Dist", "DistErr"]

ra_c  = 0.5*(ra_min + ra_max)
dec_c = 0.5*(dec_min + dec_max)
width =  (ra_max - ra_min) + 2*margin_deg
height = (dec_max - dec_min) + 2*margin_deg
center = SkyCoord(ra=ra_c*u.deg, dec=dec_c*u.deg, frame="icrs")

res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat_id)
if len(res) == 0 or len(res[0]) == 0:
    # last-chance: expand box slightly more
    res = Vizier.query_region(center, width=(width+0.4)*u.deg, height=(height+0.4)*u.deg, catalog=cat_id)
if len(res) == 0 or len(res[0]) == 0:
    raise RuntimeError("VizieR returned no GLADE rows in this region after expansion.")

glade_tab = res[0]
print(f"GLADE rows retrieved in box: {len(glade_tab)}")

# ---------- robust SkyCoord creation for GLADE RA/Dec ----------
def make_glade_sky(tab, ra_name, dec_name):
    ra_col, dec_col = tab[ra_name], tab[dec_name]
    # try plain floats in degrees
    try:
        ra_deg  = np.array(ra_col, dtype=float)
        dec_deg = np.array(dec_col, dtype=float)
        return SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    except Exception:
        pass
    # try sexagesimal strings as degrees
    try:
        return SkyCoord(ra=ra_col, dec=dec_col, unit=(u.deg, u.deg), frame="icrs")
    except Exception:
        pass
    # fall back to hourangle for RA, degrees for Dec
    return SkyCoord(ra=ra_col, dec=dec_col, unit=(u.hourangle, u.deg), frame="icrs")

des_sky   = SkyCoord(ra=des_tab["ra"]*u.deg,       dec=des_tab["dec"]*u.deg,       frame="icrs")
glade_sky = make_glade_sky(glade_tab, glade_ra, glade_dec)

# ---------- local 2″ nearest-neighbour crossmatch ----------
idx, sep2d, _ = des_sky.match_to_catalog_sky(glade_sky)
matched_mask  = sep2d < match_radius

matched_des   = des_tab[matched_mask]
matched_glade = glade_tab[idx[matched_mask]]
matched       = hstack([matched_des, matched_glade])

unmatched = des_tab[~matched_mask]

# ---------- report & save ----------
n_total    = len(des_tab)
n_matched  = len(matched)
n_unmatched= len(unmatched)
pct        = round(100.0 * n_matched / n_total, 2)

print({
    "total_DES_candidates": n_total,
    "matched_in_GLADE": n_matched,
    "unmatched": n_unmatched,
    "completeness_percent": pct,
    "match_radius_arcsec": float(match_radius.to_value(u.arcsec)),
    "glade_rows_in_box": len(glade_tab),
    "catalog_id_used": cat_id,
    "glade_ra_col": glade_ra,
    "glade_dec_col": glade_dec
})

matched.to_pandas().to_csv("GW190814_DES_vs_GLADE_matches_2arcsec.csv", index=False)
unmatched.to_pandas().to_csv("GW190814_DES_candidates_not_in_GLADE_2arcsec.csv", index=False)
print("Saved:")
print(" - GW190814_DES_vs_GLADE_matches_2arcsec.csv")
print(" - GW190814_DES_candidates_not_in_GLADE_2arcsec.csv")


Using catalog id: VII/281
GLADE RA/Dec columns: RAJ2000 DEJ2000
GLADE rows retrieved in box: 2739
{'total_DES_candidates': 589, 'matched_in_GLADE': 95, 'unmatched': 494, 'completeness_percent': 16.13, 'match_radius_arcsec': 2.0, 'glade_rows_in_box': 2739, 'catalog_id_used': 'VII/281', 'glade_ra_col': 'RAJ2000', 'glade_dec_col': 'DEJ2000'}
Saved:
 - GW190814_DES_vs_GLADE_matches_2arcsec.csv
 - GW190814_DES_candidates_not_in_GLADE_2arcsec.csv


In [11]:
# Completeness sweep for GLADE matches: 2", 3", 5" + brightness comparison at 2"

import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split("==")[0].split(">=")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for p in ["astroquery", "astropy", "pandas", "numpy"]:
    pip_install(p)

import pandas as pd, numpy as np
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

# ---------- Inputs ----------
des_csv = "/content/GW190814_candidates_90pct.csv"
# your 90% main-island rectangle (deg)
ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484
margin_deg = 0.15  # small buffer

# ---------- Load 589 DES candidates ----------
des = Table.read(des_csv, format="ascii.csv")
if not {"ra","dec"}.issubset(set(des.colnames)):
    raise ValueError("CSV must have 'ra' and 'dec' columns.")
des_sky = SkyCoord(des["ra"].astype(float)*u.deg, des["dec"].astype(float)*u.deg, frame="icrs")

# ---------- Ensure we have a GLADE subset (reuse prior glade_tab if available) ----------
def get_glade_subset():
    global glade_tab, glade_ra, glade_dec
    try:
        glade_tab, glade_ra, glade_dec  # type: ignore
        if len(glade_tab) > 0:
            return
    except Exception:
        pass
    cats = Vizier.find_catalogs("GLADE")
    if not cats:
        raise RuntimeError("No GLADE entries found on this VizieR mirror.")
    preferred = [k for k in cats.keys() if "VII/281" in k]
    cat_id = preferred[0] if preferred else list(cats.keys())[0]
    Vizier.ROW_LIMIT = 50
    sample = None
    try:
        sample = Vizier.get_catalogs(cat_id)[0]
    except Exception:
        pass
    if sample is None or len(sample.colnames) == 0:
        tiny = Vizier.query_region(SkyCoord(0*u.deg,0*u.deg), width=0.1*u.deg, height=0.1*u.deg, catalog=cat_id)
        sample = tiny[0] if (len(tiny)>0) else None
    if sample is None:
        raise RuntimeError("Could not inspect columns for GLADE.")
    ra_candidates  = ["RAJ2000","RA_ICRS","_RAJ2000","raj2000","ra"]
    dec_candidates = ["DEJ2000","DE_ICRS","_DEJ2000","dej2000","dec"]
    cols_lower = {c.lower(): c for c in sample.colnames}
    def pick(cands):
        for c in cands:
            if c.lower() in cols_lower:
                return cols_lower[c.lower()]
        return None
    glade_ra  = pick(ra_candidates)
    glade_dec = pick(dec_candidates)
    if glade_ra is None or glade_dec is None:
        raise RuntimeError(f"Could not find RA/Dec columns in GLADE. Columns={sample.colnames}")

    Vizier.ROW_LIMIT = -1
    ra_c  = 0.5*(ra_min + ra_max)
    dec_c = 0.5*(dec_min + dec_max)
    width =  (ra_max - ra_min) + 2*margin_deg
    height = (dec_max - dec_min) + 2*margin_deg
    center = SkyCoord(ra=ra_c*u.deg, dec=dec_c*u.deg, frame="icrs")
    res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat_id)
    if len(res)==0 or len(res[0])==0:
        res = Vizier.query_region(center, width=(width+0.4)*u.deg, height=(height+0.4)*u.deg, catalog=cat_id)
    if len(res)==0 or len(res[0])==0:
        raise RuntimeError("VizieR returned no GLADE rows in this region.")
    glade_tab = res[0]
    # expose names in globals
    globals()["glade_tab"] = glade_tab
    globals()["glade_ra"]  = glade_ra
    globals()["glade_dec"] = glade_dec
    print(f"GLADE rows retrieved in box: {len(glade_tab)} (RA={glade_ra}, Dec={glade_dec})")

get_glade_subset()

# ---------- Robust SkyCoord for GLADE ----------
def make_glade_sky(tab, ra_name, dec_name):
    ra_col, dec_col = tab[ra_name], tab[dec_name]
    # 1) try as float degrees
    try:
        ra_deg  = np.array(ra_col, dtype=float)
        dec_deg = np.array(dec_col, dtype=float)
        return SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    except Exception:
        pass
    # 2) try sexagesimal strings in degrees
    try:
        return SkyCoord(ra=ra_col, dec=dec_col, unit=(u.deg, u.deg), frame="icrs")
    except Exception:
        pass
    # 3) fallback: RA in hourangle, Dec in degrees
    return SkyCoord(ra=ra_col, dec=dec_col, unit=(u.hourangle, u.deg), frame="icrs")

glade_sky = make_glade_sky(glade_tab, glade_ra, glade_dec)

# ---------- Completeness function ----------
def completeness_at(radius_arcsec):
    idx, sep, _ = des_sky.match_to_catalog_sky(glade_sky)
    mask = sep < (radius_arcsec * u.arcsec)
    n_m = int(np.sum(mask))
    n_u = len(des) - n_m
    pct = round(100.0 * n_m / len(des), 2)
    return n_m, n_u, pct, mask, idx

# ---------- Sweep radii ----------
results = []
for r in [2, 3, 5]:
    n_m, n_u, pct, _, _ = completeness_at(r)
    results.append({"radius_arcsec": r, "matched": n_m, "unmatched": n_u, "percent": pct})
print("Completeness vs radius:", results)

# ---------- Brightness sanity check at 2" ----------
n_m, n_u, pct, mask2, idx2 = completeness_at(2)
des_df = des.to_pandas()
if "mag_auto_i" in des_df.columns:
    med_matched   = float(des_df.loc[mask2,  "mag_auto_i"].median())
    med_unmatched = float(des_df.loc[~mask2, "mag_auto_i"].median())
    print({"median_mag_i_matched": med_matched, "median_mag_i_unmatched": med_unmatched})
else:
    print("Note: 'mag_auto_i' not in table; skipping brightness comparison.")


Completeness vs radius: [{'radius_arcsec': 2, 'matched': 95, 'unmatched': 494, 'percent': 16.13}, {'radius_arcsec': 3, 'matched': 98, 'unmatched': 491, 'percent': 16.64}, {'radius_arcsec': 5, 'matched': 100, 'unmatched': 489, 'percent': 16.98}]
{'median_mag_i_matched': 17.457724, 'median_mag_i_unmatched': 18.8454515}


In [12]:
# 2MASS XSC crossmatch (2")
from astroquery.vizier import Vizier
from astropy.table import Table, hstack
from astropy.coordinates import SkyCoord
import astropy.units as u, numpy as np, pandas as pd

des = Table.read("/content/GW190814_candidates_90pct.csv", format="ascii.csv")
des_sky = SkyCoord(des["ra"]*u.deg, des["dec"]*u.deg)

# RA/Dec box (same as before)
ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484
margin_deg = 0.15

Vizier.ROW_LIMIT = -1
cat = "VII/233/xsc"  # 2MASS extended source catalog
ra_c, dec_c = 0.5*(ra_min+ra_max), 0.5*(dec_min+dec_max)
width, height = (ra_max-ra_min)+2*margin_deg, (dec_max-dec_min)+2*margin_deg
center = SkyCoord(ra_c*u.deg, dec_c*u.deg)

res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat)
xsc = res[0]
x_ra = np.array(xsc["RAJ2000"], dtype=float); x_dec = np.array(xsc["DEJ2000"], dtype=float)
xsc_sky = SkyCoord(x_ra*u.deg, x_dec*u.deg)

idx, sep, _ = des_sky.match_to_catalog_sky(xsc_sky)
mask = sep < 2.0*u.arcsec

print({"catalog":"2MASS XSC", "matched": int(mask.sum()), "unmatched": int(len(des)-mask.sum()),
       "percent": round(100*mask.mean(),2), "rows_in_box": len(xsc)})

hstack([des[mask], xsc[idx[mask]]]).to_pandas().to_csv("GW190814_DES_vs_2MASSXSC_2arcsec.csv", index=False)
des[~mask].to_pandas().to_csv("GW190814_DES_not_in_2MASSXSC_2arcsec.csv", index=False)


{'catalog': '2MASS XSC', 'matched': 7, 'unmatched': 582, 'percent': 1.19, 'rows_in_box': 1207}


In [13]:
# AllWISE crossmatch (3" default—WISE PSF is larger)
from astroquery.vizier import Vizier
from astropy.table import Table, hstack
from astropy.coordinates import SkyCoord
import astropy.units as u, numpy as np

des = Table.read("/content/GW190814_candidates_90pct.csv", format="ascii.csv")
des_sky = SkyCoord(des["ra"]*u.deg, des["dec"]*u.deg)

ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484
margin_deg = 0.15

Vizier.ROW_LIMIT = -1
cat = "II/328/allwise"
ra_c, dec_c = 0.5*(ra_min+ra_max), 0.5*(dec_min+dec_max)
width, height = (ra_max-ra_min)+2*margin_deg, (dec_max-dec_min)+2*margin_deg
center = SkyCoord(ra_c*u.deg, dec_c*u.deg)

res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat)
wise = res[0]
w_ra = np.array(wise["RAJ2000"], dtype=float); w_dec = np.array(wise["DEJ2000"], dtype=float)
wise_sky = SkyCoord(w_ra*u.deg, w_dec*u.deg)

idx, sep, _ = des_sky.match_to_catalog_sky(wise_sky)
mask = sep < 3.0*u.arcsec  # looser for WISE

print({"catalog":"AllWISE", "matched": int(mask.sum()), "unmatched": int(len(des)-mask.sum()),
       "percent": round(100*mask.mean(),2), "rows_in_box": len(wise)})

hstack([des[mask], wise[idx[mask]]]).to_pandas().to_csv("GW190814_DES_vs_AllWISE_3arcsec.csv", index=False)
des[~mask].to_pandas().to_csv("GW190814_DES_not_in_AllWISE_3arcsec.csv", index=False)


{'catalog': 'AllWISE', 'matched': 380, 'unmatched': 209, 'percent': 64.52, 'rows_in_box': 381870}


In [15]:
# Robust HyperLEDA match: try ICRS/J2000 columns first, then convert B1950 -> ICRS.
# Reports matches at 2", 3", 5", 10" and saves best-radius CSVs.

import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split("==")[0].split(">=")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for p in ["astroquery", "astropy", "pandas", "numpy"]:
    pip_install(p)

import numpy as np, pandas as pd
from astropy.table import Table, hstack
from astropy.coordinates import SkyCoord, FK4
import astropy.units as u
from astroquery.vizier import Vizier

# --- Inputs (your 589 and sky box) ---
des_csv = "/content/GW190814_candidates_90pct.csv"
ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484
margin_deg = 0.15

# --- Load DES ---
des = Table.read(des_csv, format="ascii.csv")
des_sky = SkyCoord(des["ra"].astype(float)*u.deg, des["dec"].astype(float)*u.deg, frame="icrs")

# --- Get HyperLEDA table on this mirror (you saw VII/237) ---
cats = Vizier.find_catalogs("HyperLEDA") or Vizier.find_catalogs("LEDA")
cat_id = [k for k in cats.keys() if ("VII/237" in k)][0] if cats else "VII/237"
print("Using catalog id:", cat_id)

# --- Pull LEDA subset in box ---
Vizier.ROW_LIMIT = -1
ra_c, dec_c = 0.5*(ra_min+ra_max), 0.5*(dec_min+dec_max)
width, height = (ra_max-ra_min)+2*margin_deg, (dec_max-dec_min)+2*margin_deg
center = SkyCoord(ra_c*u.deg, dec_c*u.deg)

res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat_id)
if len(res)==0 or len(res[0])==0:
    res = Vizier.query_region(center, width=(width+0.4)*u.deg, height=(height+0.4)*u.deg, catalog=cat_id)
if len(res)==0 or len(res[0])==0:
    raise RuntimeError("No HyperLEDA rows returned in region.")
leda = res[0]
print(f"HyperLEDA rows in box: {len(leda)}")
print("Some columns:", leda.colnames[:20])

# --- Candidate RA/Dec column name sets to try ---
# 1) ICRS/J2000 flavors
icrs_pairs = [
    ("RAJ2000","DEJ2000"),
    ("RA_ICRS","DE_ICRS"),
    ("_RAJ2000","_DEJ2000"),
    ("raj2000","dej2000"),
    ("ra","dec")
]
# 2) B1950/FK4 flavors (need conversion)
b1950_pairs = [
    ("RA1950","DE1950"),
    ("RA_B1950","DE_B1950"),
    ("RA1950h","DE1950d")   # sometimes in h/m/s + deg
]

def cols_present(tab, pair):
    return all(c in tab.colnames for c in pair)

def make_icrs(tab, ra_name, dec_name):
    # Try as float degrees; fall back to sexagesimal strings
    try:
        ra_deg  = np.array(tab[ra_name], dtype=float)
        dec_deg = np.array(tab[dec_name], dtype=float)
        return SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    except Exception:
        return SkyCoord(tab[ra_name], tab[dec_name], unit=(u.deg, u.deg), frame="icrs")

def make_fk4_to_icrs(tab, ra_name, dec_name):
    # Build FK4(B1950) coords, then transform to ICRS
    try:
        ra = np.array(tab[ra_name], dtype=float)
        de = np.array(tab[dec_name], dtype=float)
        fk4 = SkyCoord(ra=ra*u.deg, dec=de*u.deg, frame=FK4, equinox='B1950')
    except Exception:
        fk4 = SkyCoord(tab[ra_name], tab[dec_name], unit=(u.hourangle, u.deg), frame=FK4, equinox='B1950')
    return fk4.transform_to("icrs")

def try_match(glade_sky, radii_arcsec=(2,3,5,10)):
    out = []
    idx, sep, _ = des_sky.match_to_catalog_sky(glade_sky)
    for r in radii_arcsec:
        m = sep < (r*u.arcsec)
        out.append({"radius_arcsec": r, "matched": int(m.sum()), "unmatched": int(len(des)-m.sum()), "percent": round(100*m.mean(),2)})
    # Also return best at 2" with the mask and indices for saving
    m2 = sep < (2*u.arcsec)
    return out, idx, m2

results = None
used_cols = None
mode = None

# Try ICRS/J2000 columns
for ra_col, dec_col in icrs_pairs:
    if cols_present(leda, (ra_col, dec_col)):
        try:
            sky = make_icrs(leda, ra_col, dec_col)
            results, idx, m2 = try_match(sky)
            print(f"Used ICRS/J2000 columns: {ra_col}, {dec_col}")
            break
        except Exception as e:
            print(f"ICRS build failed for ({ra_col},{dec_col}):", e)

# If still none, try B1950→ICRS
if results is None:
    for ra_col, dec_col in b1950_pairs:
        if cols_present(leda, (ra_col, dec_col)):
            try:
                sky = make_fk4_to_icrs(leda, ra_col, dec_col)
                results, idx, m2 = try_match(sky)
                print(f"Converted B1950→ICRS from columns: {ra_col}, {dec_col}")
                break
            except Exception as e:
                print(f"B1950 build failed for ({ra_col},{dec_col}):", e)

if results is None:
    raise RuntimeError("Could not construct HyperLEDA sky coordinates from available columns.")

print("Completeness vs radius:", results)

# Save matched/unmatched at 2"
matched = hstack([des[m2], leda[idx[m2]]])
unmatched = des[~m2]
matched.to_pandas().to_csv("GW190814_DES_vs_HyperLEDA_matches_2arcsec.csv", index=False)
unmatched.to_pandas().to_csv("GW190814_DES_candidates_not_in_HyperLEDA_2arcsec.csv", index=False)
print("Saved CSVs for 2\" radius.")


Using catalog id: VII/237
HyperLEDA rows in box: 1199
Some columns: ['RAJ2000', 'DEJ2000', 'PGC']
Used ICRS/J2000 columns: RAJ2000, DEJ2000
Completeness vs radius: [{'radius_arcsec': 2, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 3, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 5, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 10, 'matched': 0, 'unmatched': 589, 'percent': 0.0}]
Saved CSVs for 2" radius.


In [16]:
# Rebuild HyperLEDA SkyCoord robustly (auto-detect hourangle) and recompute matches

import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import hstack

# Expect these are already in memory from your prior cell:
#   des (Table with 'ra','dec'), des_sky (ICRS),
#   leda (HyperLEDA subset Table), leda_ra, leda_dec

def smart_sky_icrs(tab, ra_name, dec_name):
    ra_col, dec_col = tab[ra_name], tab[dec_name]
    ra_arr = np.asarray(ra_col)
    dec_arr = np.asarray(dec_col)

    # Detect if RA looks like sexagesimal (contains ':')
    ra_unit = u.deg
    if ra_arr.dtype.kind in ("U","S","O"):
        try:
            # sample a few entries
            sample = "".join(str(ra_arr[i]) for i in range(min(3, len(ra_arr))))
            if ":" in sample:  # likely h:m:s
                ra_unit = u.hourangle
        except Exception:
            pass

    # Detect if Dec looks like sexagesimal; default to degrees either way
    dec_unit = u.deg
    if dec_arr.dtype.kind in ("U","S","O"):
        try:
            sample = "".join(str(dec_arr[i]) for i in range(min(3, len(dec_arr))))
            if ":" in sample:
                dec_unit = u.deg  # sexagesimal deg string
        except Exception:
            pass

    # Build SkyCoord with detected units
    return SkyCoord(ra=ra_col, dec=dec_col, unit=(ra_unit, dec_unit), frame="icrs")

leda_sky = smart_sky_icrs(leda, leda_ra, leda_dec)

def completeness_at(radius_arcsec):
    idx, sep, _ = des_sky.match_to_catalog_sky(leda_sky)
    m = sep < (radius_arcsec * u.arcsec)
    return int(m.sum()), int(len(des) - int(m.sum())), round(100*float(m.mean()), 2), m, idx

results = []
for r in [2, 3, 5, 10]:
    n_m, n_u, pct, _, _ = completeness_at(r)
    results.append({"radius_arcsec": r, "matched": n_m, "unmatched": n_u, "percent": pct})
print("HyperLEDA completeness vs radius:", results)

# Save 2" outputs
n_m, n_u, pct, mask2, idx2 = completeness_at(2)
matched = hstack([des[mask2], leda[idx2[mask2]]])
unmatched = des[~mask2]
matched.to_pandas().to_csv("GW190814_DES_vs_HyperLEDA_matches_2arcsec.csv", index=False)
unmatched.to_pandas().to_csv("GW190814_DES_candidates_not_in_HyperLEDA_2arcsec.csv", index=False)
print("Saved 2\" CSVs.")


HyperLEDA completeness vs radius: [{'radius_arcsec': 2, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 3, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 5, 'matched': 0, 'unmatched': 589, 'percent': 0.0}, {'radius_arcsec': 10, 'matched': 0, 'unmatched': 589, 'percent': 0.0}]
Saved 2" CSVs.


In [18]:
# Robust HyperLEDA matching: try RA as hourangle *and* as degrees, fix arcsec printing

import numpy as np
from astropy.table import hstack
from astropy.coordinates import SkyCoord, FK4
import astropy.units as u

# Assumes you already have in memory:
#   des (Table with 'ra','dec'), des_sky (ICRS),
#   leda  (HyperLEDA subset Table), leda_ra, leda_dec

def build_icrs_hourangle(tab, ra_name, dec_name):
    # RA as hourangle, Dec as degrees
    return SkyCoord(ra=tab[ra_name], dec=tab[dec_name],
                    unit=(u.hourangle, u.deg), frame="icrs")

def build_icrs_degrees(tab, ra_name, dec_name):
    # RA, Dec both in degrees
    ra = np.array(tab[ra_name], dtype=float)
    de = np.array(tab[dec_name], dtype=float)
    return SkyCoord(ra=ra*u.deg, dec=de*u.deg, frame="icrs")

def completeness_report(cat_sky, label):
    idx, sep, _ = des_sky.match_to_catalog_sky(cat_sky)
    out = []
    for r in [2, 3, 5, 10]:
        m = sep < (r*u.arcsec)
        out.append({"mode": label, "radius_arcsec": r,
                    "matched": int(m.sum()),
                    "unmatched": len(des)-int(m.sum()),
                    "percent": round(100*float(m.mean()), 2)})
    min_sep_arcsec = float(sep.to(u.arcsec).min().value)
    return out, idx, sep, min_sep_arcsec

# Try RA hourangle
try:
    leda_h = build_icrs_hourangle(leda, leda_ra, leda_dec)
    rep_h, idx_h, sep_h, min_h = completeness_report(leda_h, "RA=hourangle")
    print("Hourangle parse min separation (arcsec):", min_h)
    print("Completeness:", rep_h)
except Exception as e:
    rep_h, idx_h, sep_h, min_h = ([], None, None, np.inf)
    print("Hourangle parse failed:", e)

# Try RA degrees
try:
    leda_d = build_icrs_degrees(leda, leda_ra, leda_dec)
    rep_d, idx_d, sep_d, min_d = completeness_report(leda_d, "RA=degrees")
    print("Degrees parse min separation (arcsec):", min_d)
    print("Completeness:", rep_d)
except Exception as e:
    rep_d, idx_d, sep_d, min_d = ([], None, None, np.inf)
    print("Degrees parse failed:", e)

# Choose the parse with smaller min separation (i.e., closer to real)
if min_h < min_d:
    chosen_label, chosen_idx, chosen_sep = "hourangle", idx_h, sep_h
    print("Using RA=hourangle interpretation.")
else:
    chosen_label, chosen_idx, chosen_sep = "degrees", idx_d, sep_d
    print("Using RA=degrees interpretation.")

# Save 2" results if any
if chosen_idx is not None:
    m2 = chosen_sep < (2*u.arcsec)
    matched = hstack([des[m2], leda[chosen_idx[m2]]])
    unmatched = des[~m2]
    matched.to_pandas().to_csv("GW190814_DES_vs_HyperLEDA_matches_2arcsec.csv", index=False)
    unmatched.to_pandas().to_csv("GW190814_DES_candidates_not_in_HyperLEDA_2arcsec.csv", index=False)
    print("Saved 2\" CSVs (mode:", chosen_label, ").")
else:
    print("No valid HyperLEDA parse available to save matches.")


Hourangle parse min separation (arcsec): 0.07579758084049534
Completeness: [{'mode': 'RA=hourangle', 'radius_arcsec': 2, 'matched': 51, 'unmatched': 538, 'percent': 8.66}, {'mode': 'RA=hourangle', 'radius_arcsec': 3, 'matched': 53, 'unmatched': 536, 'percent': 9.0}, {'mode': 'RA=hourangle', 'radius_arcsec': 5, 'matched': 54, 'unmatched': 535, 'percent': 9.17}, {'mode': 'RA=hourangle', 'radius_arcsec': 10, 'matched': 54, 'unmatched': 535, 'percent': 9.17}]
Degrees parse failed: could not convert string to float: '00 39 07.8'
Using RA=hourangle interpretation.
Saved 2" CSVs (mode: hourangle ).


In [20]:
# Pan-STARRS DR2 crossmatch (1") — II/349/ps1 (MeanObject table)
import sys, subprocess
def pip_install(pkg):
    try: __import__(pkg.split("==")[0].split(">=")[0])
    except Exception: subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for p in ["astroquery", "astropy", "pandas", "numpy"]:
    pip_install(p)

import numpy as np, pandas as pd
from astropy.table import Table, hstack
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

# Inputs
des_csv = "/content/GW190814_candidates_90pct.csv"
ra_min, ra_max = 10.2392578125, 14.8974609375
dec_min, dec_max = -27.3215904756, -22.2256480484
margin_deg = 0.15
match_radius = 1.0 * u.arcsec   # PS1 astrometry is tight

# Load DES candidates
des = Table.read(des_csv, format="ascii.csv")
if not {"ra","dec"}.issubset(set(des.colnames)):
    raise ValueError("CSV must have 'ra' and 'dec' columns.")
des_sky = SkyCoord(des["ra"].astype(float)*u.deg, des["dec"].astype(float)*u.deg, frame="icrs")

# Region box
ra_c, dec_c = 0.5*(ra_min+ra_max), 0.5*(dec_min+dec_max)
width, height = (ra_max-ra_min)+2*margin_deg, (dec_max-dec_min)+2*margin_deg
center = SkyCoord(ra_c*u.deg, dec_c*u.deg)

# Query PS1 subset
Vizier.ROW_LIMIT = -1
cat = "II/349/ps1"
res = Vizier.query_region(center, width=width*u.deg, height=height*u.deg, catalog=cat)
if len(res)==0 or len(res[0])==0:
    res = Vizier.query_region(center, width=(width+0.4)*u.deg, height=(height+0.4)*u.deg, catalog=cat)
if len(res)==0 or len(res[0])==0:
    raise RuntimeError("No Pan-STARRS rows returned in region.")
ps1 = res[0]
# RA/Dec columns
ra_name = "RAJ2000" if "RAJ2000" in ps1.colnames else "RAICRS" if "RAICRS" in ps1.colnames else list(ps1.colnames)[0]
dec_name= "DEJ2000" if "DEJ2000" in ps1.colnames else "DEICRS" if "DEICRS" in ps1.colnames else list(ps1.colnames)[1]
ps1_sky = SkyCoord(np.array(ps1[ra_name], dtype=float)*u.deg, np.array(ps1[dec_name], dtype=float)*u.deg, frame="icrs")

# Local match
idx, sep, _ = des_sky.match_to_catalog_sky(ps1_sky)
mask = sep < match_radius

print({"catalog":"Pan-STARRS DR2", "matched": int(mask.sum()), "unmatched": int(len(des)-mask.sum()),
       "percent": round(100*mask.mean(),2), "rows_in_box": len(ps1)})

hstack([des[mask], ps1[idx[mask]]]).to_pandas().to_csv("GW190814_DES_vs_PS1_1arcsec.csv", index=False)
des[~mask].to_pandas().to_csv("GW190814_DES_not_in_PS1_1arcsec.csv", index=False)
print("Saved CSVs for PS1 (1\").")


{'catalog': 'Pan-STARRS DR2', 'matched': 431, 'unmatched': 158, 'percent': 73.17, 'rows_in_box': 529700}
Saved CSVs for PS1 (1").
